### Setup Environment:

In [ ]:
from src.embeddings import get_embeddings_df, load_data, split_dataset, train_and_evaluate_model

# Class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Models
# Random forest
from sklearn.ensemble import RandomForestClassifier
# Logistic regression
from sklearn.linear_model import LogisticRegression
# Support vector machine
from sklearn.svm import SVC
# Decision tree
from sklearn.tree import DecisionTreeClassifier

## Embeddings Generation

* **Batch Size:** Images per batch to convert to embeddings (Adjust depending on your memory)

* **Path:** Path to the images

* **Output Directory:** Directory to save the embeddings

* **Backbone:** Select a backbone from the list of possible backbones:
    * 'dinov2_small'
    * 'dinov2_base'
    * 'dinov2_large'
    * 'dinov2_giant'
    * 'clip_base',
    * 'clip_large',
    * 'convnextv2_tiny'
    * 'convnextv2_base'
    * 'convnextv2_large'
    * 'convnext_tiny'
    * 'convnext_small'
    * 'convnext_base'
    * 'convnext_large'
    * 'swin_tiny'
    * 'swin_small'
    * 'swin_base'
    * 'vit_base'
    * 'vit_large'
    * 'retfound'

In [ ]:
# Foundational Models
dino_backbone = ['dinov2_small', 'dinov2_base', 'dinov2_large', 'dinov2_giant']

clip_backbone = ['clip_base', 'clip_large']

# ImageNet:

### Convnext
convnext_backbone = ['convnextv2_tiny', 'convnextv2_base', 'convnextv2_large'] + ['convnext_tiny', 'convnext_small', 'convnext_base', 'convnext_large']

### Swin Transformer
swin_transformer_backbone = ['swin_tiny', 'swin_small', 'swin_base']

### ViT
vit_backbone = ['vit_base', 'vit_large']

retfound_backbone = ['retfound']

backbones = dino_backbone + clip_backbone + convnext_backbone + swin_transformer_backbone + vit_backbone + retfound_backbone

backbones

In [ ]:
batch_size = 32
path = 'data/images/'
backbone = 'dinov2_base'
out_dir = 'Embeddings'

get_embeddings_df(batch_size=batch_size, path=path, backbone=backbone, directory=out_dir)

## Evaluate the Embeddings

In [ ]:
def run_experiments(LABELS_PATH, LABEL, EMBEDDINGS_BACKBONE, EMBEDDINGS_DIR, TEST_SIZE, CLASS_WEIGTHS, NORMAL=False):
    
    # Get the dataset
    X, y = load_data(labels_path=LABELS_PATH, backbone=EMBEDDINGS_BACKBONE, label=LABEL, directory=EMBEDDINGS_DIR, normal=NORMAL)

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = split_dataset(X, y, test_size=TEST_SIZE)

    # Define a list of models to test
    if CLASS_WEIGTHS:
        class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
        # Create a dictionary from class labels and weights
        class_weights = {class_label: weight for class_label, weight in zip(np.unique(y_train), class_weights)}
        print(f'Setting class weigths to: {class_weights}')

        models = [
            #("Random Forest", RandomForestClassifier(class_weight=class_weights)),
            #('Decision Tree', DecisionTreeClassifier(class_weight=class_weights)),
            ("SVM", SVC(class_weight=class_weights)),
            ("Logistic Regression", LogisticRegression(class_weight=class_weights))
        ]
    else:
        models = [
            #("Random Forest", RandomForestClassifier()),
            #('Decision Tree', DecisionTreeClassifier()),
            ("SVM", SVC()),
            ("Logistic Regression", LogisticRegression())
        ]

    # Run the experiments
    train_and_evaluate_model(X_train, X_test, y_train, y_test, models=models)

#### Constants

In [ ]:
# Constants:
LABELS_PATH = 'data/labels.csv'
EMBEDDINGS_BACKBONE = 'dinov2_base'
EMBEDDINGS_DIR = 'Embeddings'
TEST_SIZE = 0.3
CLASS_WEIGTHS = True

### Diabetes

In [ ]:
NORMAL = False
LABEL = 'diabetes'

In [ ]:
run_experiments(LABELS_PATH, LABEL, EMBEDDINGS_BACKBONE, EMBEDDINGS_DIR, TEST_SIZE, CLASS_WEIGTHS, NORMAL=NORMAL)

### Sex

In [ ]:
# Constants:
LABEL = 'patient_sex'

In [ ]:
run_experiments(LABELS_PATH, LABEL, EMBEDDINGS_BACKBONE, EMBEDDINGS_DIR, TEST_SIZE, CLASS_WEIGTHS)

### Diabetic Retinopathy

In [ ]:
LABEL = 'diabetic_retinopathy'

In [ ]:
run_experiments(LABELS_PATH, LABEL, EMBEDDINGS_BACKBONE, EMBEDDINGS_DIR, TEST_SIZE, CLASS_WEIGTHS)